## 💰Case 5: Flight Price Prediction
**Objective:** To understand how these parameters influence price fluctuations and to build a data-driven dynamic pricing system that helps airlines and travelers anticipate fare changes in advance.

### Step 1: Load & inspect data

In [0]:
# Step 1 — Load & Inspect
from pyspark.sql.functions import col, trim

# Try table first, else read from FileStore (update path if necessary)
try:
    df = spark.table("workspace.default.`5_flight_price_prediction`")
    print("Loaded table: workspace.default.`5_flight_price_prediction`")
except Exception as e:
    print("Table not found — falling back to FileStore CSV. Error:", e)
    path = "/FileStore/tables/5_flight_price_prediction.csv"   # << update if different
    df = spark.read.option("header", True).option("inferSchema", True).csv(path)
    print("Loaded CSV from:", path)

# Inspect
df.printSchema()
display(df.limit(5))


Loaded table: workspace.default.`5_flight_price_prediction`
root
 |-- airline: string (nullable = true)
 |-- flight: string (nullable = true)
 |-- source_city: string (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- stops: string (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- class: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- days_left: long (nullable = true)
 |-- price: long (nullable = true)



airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


### Step 2: Clean column names & basic cleaning

In [0]:
# Step 2 — Clean names, trim strings, drop exact nulls for target
import re
from pyspark.sql.functions import trim

def clean_column_name(name):
    return re.sub(r'[^A-Za-z0-9]+', '_', name.strip())

df = df.toDF(*[clean_column_name(c) for c in df.columns])
df = df.select([trim(col(c)).alias(c) for c in df.columns])

# Ensure price is numeric and drop rows missing price
df = df.withColumn("price", col("price").cast("double"))
df = df.withColumn("duration", col("duration").cast("double"))
df = df.withColumn("days_left", col("days_left").cast("long"))

df = df.filter(col("price").isNotNull())
print("Rows after drop null price:", df.count())
display(df.limit(5))


Rows after drop null price: 300153


airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953.0
SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953.0
AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956.0
Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955.0
Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955.0


### Step 3: Quick SQL EDA

In [0]:
# Step 3 — SQL EDA
df.createOrReplaceTempView("prices")

# average price by airline
spark.sql("""
  SELECT airline, COUNT(*) AS cnt, ROUND(AVG(price),2) AS avg_price
  FROM prices
  GROUP BY airline
  ORDER BY avg_price DESC
  LIMIT 10
""").show()

# price vs class
spark.sql("""
  SELECT class, COUNT(*) AS cnt, ROUND(AVG(price),2) AS avg_price
  FROM prices
  GROUP BY class
""").show()

# days_left buckets — price vs days_left
spark.sql("""
  SELECT days_left, COUNT(*) AS cnt, ROUND(AVG(price),2) AS avg_price
  FROM prices
  GROUP BY days_left
  ORDER BY days_left
  LIMIT 20
""").show()


+---------+------+---------+
|  airline|   cnt|avg_price|
+---------+------+---------+
|  Vistara|127859| 30396.54|
|Air_India| 80892| 23507.02|
| SpiceJet|  9011|  6179.28|
| GO_FIRST| 23173|  5652.01|
|   Indigo| 43120|  5324.22|
|  AirAsia| 16098|  4091.07|
+---------+------+---------+

+--------+------+---------+
|   class|   cnt|avg_price|
+--------+------+---------+
| Economy|206666|  6572.34|
|Business| 93487| 52540.08|
+--------+------+---------+

+---------+----+---------+
|days_left| cnt|avg_price|
+---------+----+---------+
|        1|1927| 21591.87|
|        2|4026|  30211.3|
|        3|4248| 28976.08|
|        4|5077| 25730.91|
|        5|5392| 26679.77|
|        6|5740| 24856.49|
|        7|5703| 25588.37|
|        8|5767| 24895.88|
|        9|5665| 25726.25|
|       10|5822| 25572.82|
|       11|6417| 22990.66|
|       12|6381|  22505.8|
|       13|6404| 22498.89|
|       14|6349|  22678.0|
|       15|6340| 21952.54|
|       16|6272| 20503.55|
|       17|6419| 20386.35|


### Step 4: Feature engineering
**Categorical features:** airline, source_city, departure_time, stops, arrival_time, destination_city, class        
**Numeric features:** duration, days_left

In [0]:
# Step 4 — Feature engineering (Databricks Free Edition Safe)

from pyspark.sql.functions import col, coalesce, lit, when  # 👈 Added 'when' import

# Select only the columns relevant for ML
df_model = df.select(
    "airline","source_city","departure_time","stops","arrival_time",
    "destination_city","class","duration","days_left","price"
)

# Fill nulls in categoricals with 'unknown'
cat_cols = ["airline","source_city","departure_time","stops","arrival_time","destination_city","class"]
for c in cat_cols:
    df_model = df_model.withColumn(c, coalesce(col(c), lit("unknown")))

# Replace weird 'stops' text like "zero" → "0" to ensure numeric compatibility
df_model = df_model.withColumn(
    "stops",
    when(col("stops").rlike("(?i)zero|0"), "0").otherwise(col("stops"))
)

display(df_model.limit(5))
print("✅ Step 4 complete — Clean categorical + numeric features ready")


airline,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
SpiceJet,Delhi,Evening,0,Night,Mumbai,Economy,2.17,1,5953.0
SpiceJet,Delhi,Early_Morning,0,Morning,Mumbai,Economy,2.33,1,5953.0
AirAsia,Delhi,Early_Morning,0,Early_Morning,Mumbai,Economy,2.17,1,5956.0
Vistara,Delhi,Morning,0,Afternoon,Mumbai,Economy,2.25,1,5955.0
Vistara,Delhi,Morning,0,Morning,Mumbai,Economy,2.33,1,5955.0


✅ Step 4 complete — Clean categorical + numeric features ready


### Step 5: Build ML pipeline & split data

In [0]:
# Step 5 — Build Pipeline (OneHot + VectorAssembler + StandardScaler + LinearRegression)
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression

# 1) index categorical columns
indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in cat_cols]

# 2) OHE for indexed cats (OneHotEncoder supports multiple inputs)
ohe = OneHotEncoder(inputCols=[c + "_idx" for c in cat_cols],
                    outputCols=[c + "_ohe" for c in cat_cols],
                    handleInvalid="keep")

# 3) assemble features
feature_inputs = [c + "_ohe" for c in cat_cols] + ["duration","days_left"]
assembler = VectorAssembler(inputCols=feature_inputs, outputCol="features_vec", handleInvalid="keep")

# 4) scale (optional)
scaler = StandardScaler(inputCol="features_vec", outputCol="features", withStd=True, withMean=False)

# 5) model
lr = LinearRegression(featuresCol="features", labelCol="price", maxIter=50)

pipeline = Pipeline(stages=indexers + [ohe, assembler, scaler, lr])

# Train/test split
train, test = df_model.randomSplit([0.8,0.2], seed=42)
print("Train:", train.count(), "Test:", test.count())


Train: 240288 Test: 59865


### Step 6: Train model

In [0]:
# Step 6 — Fit the pipeline
model = pipeline.fit(train)
print("✅ Model trained")


✅ Model trained


### Step 7: Predict & evaluate

In [0]:
# Step 7 — Predict and evaluate
pred = model.transform(test)
display(pred.select("airline","source_city","destination_city","price","prediction").limit(10))

from pyspark.ml.evaluation import RegressionEvaluator
evaluator_rmse = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(pred)
r2 = evaluator_r2.evaluate(pred)
print(f"RMSE: {rmse:.2f}   R2: {r2:.3f}")


airline,source_city,destination_city,price,prediction
AirAsia,Bangalore,Chennai,3499.0,-1868.6742421315466
AirAsia,Bangalore,Chennai,1715.0,-2258.7291436559935
AirAsia,Bangalore,Chennai,1822.0,-2518.765744672288
AirAsia,Bangalore,Chennai,1822.0,-3558.9121487374723
AirAsia,Bangalore,Chennai,1822.0,-3818.9487497537702
AirAsia,Bangalore,Chennai,1822.0,-3948.967050261919
AirAsia,Bangalore,Chennai,1603.0,-4078.985350770068
AirAsia,Bangalore,Chennai,1603.0,-4339.021951786366
AirAsia,Bangalore,Chennai,1603.0,-4729.076853310813
AirAsia,Bangalore,Chennai,1603.0,-5249.150055343402


RMSE: 6797.33   R2: 0.910


### Step 8: Save predictions to SQL table (for visualization) & save model to DBFS

In [0]:
# Step 8 — Save predictions as SQL table (overwrite)
pred.select("airline","source_city","destination_city","class","days_left","duration","price","prediction") \
    .write.mode("overwrite").saveAsTable("flight_price_predictions")

print("✅ Predictions saved as SQL table: flight_price_predictions")


✅ Predictions saved as SQL table: flight_price_predictions


### Step 9: SQL visualization

In [0]:
%sql
-- Average predicted price by airline
SELECT airline, ROUND(AVG(prediction),2) AS avg_pred_price, ROUND(AVG(price),2) AS avg_actual_price
FROM flight_price_predictions
GROUP BY airline
ORDER BY avg_pred_price DESC;

-- Price vs days_left: show avg predicted price for early vs late booking
SELECT days_left, ROUND(AVG(prediction),2) AS avg_pred_price, COUNT(*) as cnt
FROM flight_price_predictions
GROUP BY days_left
ORDER BY days_left
LIMIT 30;


days_left,avg_pred_price,cnt
1,16052.5,395
2,25142.01,773
3,23705.42,841
4,25258.77,984
5,26370.8,1065
6,24445.73,1106
7,25211.0,1102
8,24206.81,1147
9,26209.41,1190
10,22685.04,1194


Databricks visualization. Run in Databricks to view.